<a href="https://colab.research.google.com/github/qkrsj/AI-PROGRAMMING/blob/main/LG_aimers/W8A8%20GPTQ/LGAimers_Phase2_Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import

In [ ]:
!pip install llmcompressor

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 282.1/282.1 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-ml-py
    Found existing installation: nvidia-ml-py 13.590.48
    Uninstalling nvidia-ml-py-13.590.48:
      Successfully uninstalled nvidia-ml-py-13.590.48
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.2
    Uninstalling tqdm

In [ ]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# Now these imports will see the version installed by llmcompressor
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor.modifiers.awq import AWQModifier

In [ ]:
from huggingface_hub import snapshot_download

# 허깅페이스에서 코랩 로컬 디스크(/content)로 직접 다운로드
model_path = snapshot_download(
    repo_id="LGAI-EXAONE/EXAONE-4.0-1.2B",
    local_dir="./base_model",
    ignore_patterns=["*.msgpack", "*.h5"] # 불필요한 파일 제외
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.56G [00:00<?, ?B/s]

# Setting

In [ ]:
MODEL_ID = "./base_model"
OUT_DIR  = "./model"

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 256
MAX_SEQUENCE_LENGTH = 512

# Quantization
SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE  = ["embed_tokens", "lm_head"]

# Model Loads

In [ ]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
)

print("[INFO] 모델/토크나이저 로드 완료")

[INFO] 모델 로드 중...
[INFO] 모델/토크나이저 로드 완료


# Dataset Loads & Preprocess

In [ ]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False)
    }

ds = ds.map(preprocess)

print("[INFO] 데이터 전처리 완료")

[INFO] 캘리브레이션 데이터 로드 중...


README.md: 0.00B [00:00, ?B/s]

data/train.parquet:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Map:   0%|          | 0/256 [00:00<?, ? examples/s]

[INFO] 데이터 전처리 완료


# Quantization

## GPTQ

In [ ]:
# Quantization
SCHEME = "W8A8"
TARGETS = ["Linear"]
IGNORE  = ["embed_tokens", "lm_head"]

print(f"[INFO] GPTQ 시작 (scheme={SCHEME}, samples={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH})...")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] GPTQ 완료")

[INFO] GPTQ 시작 (scheme=W8A8, samples=256, max_len=512)...


Tokenizing:   0%|          | 0/256 [00:00<?, ? examples/s]

2026-02-06T13:58:06.861279+0000 | reset | INFO - Compression lifecycle reset
2026-02-06T13:58:06.863439+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-06T13:58:06.899310+0000 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-06T13:58:06.899893+0000 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`


(1/31): Calibrating: 100%|██████████| 256/256 [00:03<00:00, 84.61it/s] 

2026-02-06T13:58:10.578588+0000 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 256 samples


...

(30/31): Calibrating: 100%|██████████| 256/256 [00:02<00:00, 116.32it/s]

2026-02-06T14:03:39.543079+0000 | compress_modules | INFO - Quantizing model.layers.29.self_attn.q_proj using 256 samples


2026-02-06T14:03:40.575884+0000 | compress | METRIC - time 1.03s
2026-02-06T14:03:40.577272+0000 | compress | METRIC - error 37.68
2026-02-06T14:03:40.578003+0000 | compress | METRIC - GPU 0 | usage: 5.96% | total memory: 24 GB
2026-02-06T14:03:40.578625+0000 | compress | METRIC - Compressed module size: 8.394752 MB
2026-02-06T14:03:40.579532+0000 | compress_modules | INFO - Quantizing model.layers.29.self_attn.k_proj using 256 samples
2026-02-06T14:03:41.602528+0000 | compress | METRIC - time 1.02s
2026-02-06T14:03:41.603915+0000 | compress | METRIC - error 13.80
2026-02-06T14:03:41.604679+0000 | compress | METRIC - GPU 0 | usage: 5.96% | total memory: 24 GB
2026-02-06T14:03:41.605154+0000 | compress | METRIC - Compressed module size: 2.098688 MB
2026-02-06T14:03:41.606228+0000 | compress_modules | INFO - Quantizing model.layers.29.self_attn.v_proj using 256 samples
2026-02-06T14:03:42.616740+0000 | compress | METRIC - time 1.01s
2026-02-06T14:03:42.618188+0000 | compress | METRIC - e

(31/31): Propagating: 100%|██████████| 256/256 [00:00<00:00, 1102.82it/s]

2026-02-06T14:03:49.147861+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-06T14:03:49.195993+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
[INFO] GPTQ 완료


L4 기준, GPTQ 양자화 기법 적용 시 6분 소요

## AWQ

In [ ]:
print(f"[INFO] AWQ 시작 (scheme={SCHEME}, samples={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH})...")

recipe = [
    AWQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] AWQ 완료")

[INFO] AWQ 시작 (scheme=W4A16, samples=256, max_len=512)...


Tokenizing:   0%|          | 0/256 [00:00<?, ? examples/s]

2026-02-06T07:30:42.817529+0000 | reset | INFO - Compression lifecycle reset
2026-02-06T07:30:42.819803+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-06T07:30:42.857950+0000 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...
2026-02-06T07:30:42.859843+0000 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.0.self_attn.v_proj for mapping AWQMapping(smooth_layer='re:.*v_proj$', balance_layers=['re:.*o_proj$']) because found incompatible balance layers
2026-02-06T07:30:42.860590+0000 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.1.self_attn.v_proj for mapping AWQMapping(smooth_layer='re:.*v_proj$', balance_layers=['re:.*o_proj$']) because found incompatible balance layers
2026-02-06T07:30:42.861263+0000 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.2.self_attn.v_proj for mapping AWQMapping(smooth_layer='re:.*v_proj$', balance_layers=['re:.*o_proj$']) because found incompatible

(31/31): Calibrating: 100%|██████████| 256/256 [00:00<00:00, 839.05it/s]
Smoothing: 0it [00:00, ?it/s]
(31/31): Propagating: 100%|██████████| 256/256 [00:00<00:00, 1078.24it/s]
Smoothing: 0it [00:00, ?it/s]
Calibrating weights: 210it [00:02, 102.79it/s]

2026-02-06T07:32:26.890218+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-06T07:32:26.943801+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
[INFO] AWQ 완료


# Model Save

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"[INFO] 모델 저장 완료: {OUT_DIR}")

2026-02-06T14:04:22.826472+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 210it [00:01, 125.29it/s]


[INFO] 모델 저장 완료: ./model


In [ ]:
import os
from pathlib import Path

def get_dir_size(path, extension=".safetensors"):
    """특정 확장자를 가진 파일들의 총 용량을 GB 단위로 반환"""
    total_size = 0
    for file in Path(path).rglob(f"*{extension}"):
        total_size += os.path.getsize(file)
    return total_size / (1024**3)  # GB 단위 변환

# 1. 경로 설정 (기존 모델 경로를 적어주세요)
ORIGINAL_MODEL_PATH = "./base_model" # 혹은 "LGAI-EXAONE/EXAONE-4.0-1.2B"

# 2. 용량 계산
original_size = get_dir_size(ORIGINAL_MODEL_PATH)
quantized_size = get_dir_size(OUT_DIR)

# 3. 결과 출력
print("-" * 30)
print(f"📊 용량 비교 결과 (Unit: GB)")
print(f"· 원본 모델: {original_size:.2f} GB")
print(f"· 양자화 모델: {quantized_size:.2f} GB")
print(f"· 압축률: {(1-quantized_size/original_size)*100:.1f}% 절감")
print("-" * 30)

if quantized_size > original_size:
    print("⚠️ 경고: 양자화 후 용량이 더 커졌습니다. save_compressed=True 설정을 다시 확인하세요.")

------------------------------
📊 용량 비교 결과 (Unit: GB)
· 원본 모델: 2.38 GB
· 양자화 모델: 1.78 GB
· 압축률: 25.4% 절감
------------------------------


# Submission

In [ ]:
zip_name = "baseline_submit"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

print(f"[INFO] 생성 완료: {zip_name}.zip")

[INFO] baseline_submit.zip 생성 중...
[INFO] 생성 완료: baseline_submit.zip


# 추론 엔진 적용 - vLLM

In [ ]:
!pip install vllm
from vllm import LLM, SamplingParams

prompts = [
    [{"role":"user", "content":"Explain how wonderful you are"}],
    [{"role":"user", "content":"너가 얼마나 대단한지 설명해 봐"}]
]
sampling_params = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=256)

llm = LLM(model="EXAONE-4.0-1.2B-GPTQ")

outputs = llm.chat(prompts, sampling_params)

for output in outputs:
    print("###########")
    print(output.outputs[0].text)
    print()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 143.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 132.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 138.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.9/34.9 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3

# Evaluation

In [ ]:
!pip install lm-eval[vllm]
!pip install vllm
!pip install hf_transfer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 155.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 135.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 103.2 MB/s eta 0:00:00
   ━━━

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.4 MB/s eta 0:00:00


In [ ]:
# 1. 경로 설정 (방금 저장한 양자화 모델 경로)
MODEL_PATH = OUT_DIR # 본인의 OUT_DIR 경로로 수정하세요.

!lm_eval --model vllm \
    --model_args pretrained={MODEL_PATH},gpu_memory_utilization=0.8,enable_thinking=False,max_gen_toks=2048 \
    --tasks gsm8k \
    --limit 512 \
    --output_path ./results \
    --apply_chat_template \
    --batch_size auto

2026-02-06:08:12:41 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-02-06:08:12:41 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-06:08:12:46 INFO     [_cli.run:376] Selected Tasks: ['gsm8k']
2026-02-06:08:12:48 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-06:08:12:48 INFO     [evaluator:236] Initializing vllm model, with arguments: {'pretrained': './model', 'gpu_memory_utilization': 0.8, 'enable_thinking': False, 'max_gen_toks': 2048}
2026-02-06 08:12:54.591197: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770365574.615947   14882 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to regis

In [ ]:
# 1. 경로 설정 (방금 저장한 양자화 모델 경로)
MODEL_PATH = MODEL_ID # 본인의 OUT_DIR 경로로 수정하세요.

!lm_eval --model vllm \
    --model_args pretrained={MODEL_PATH},gpu_memory_utilization=0.8,enable_thinking=False,max_gen_toks=2048 \
    --tasks gsm8k \
    --limit 512 \
    --output_path ./results \
    --apply_chat_template \
    --batch_size auto

2026-02-05:12:45:26 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-02-05:12:45:26 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-05:12:45:31 INFO     [_cli.run:376] Selected Tasks: ['gsm8k']
2026-02-05:12:45:33 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-05:12:45:33 INFO     [evaluator:236] Initializing vllm model, with arguments: {'pretrained': './base_model', 'gpu_memory_utilization': 0.8, 'enable_thinking': False, 'max_gen_toks': 2048}
2026-02-05 12:45:39.853145: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770295539.877324   18306 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to 